# FastMCP Apps 실습 예제 (Colab)

이 노트북은 FastMCP Apps 문서를 바탕으로 만든 **Colab 실습 예제**입니다.

핵심 개념:
- FastMCP Apps는 **대화 안에 렌더링되는 인터랙티브 UI**를 만들기 위한 기능입니다.
- 가장 빠른 시작은 **`@mcp.tool(app=True)` + `PrefabApp` 반환** 방식입니다.
- 서버-상호작용이 많은 앱은 **`FastMCPApp`** 으로
  - `@app.ui()` : 모델이 호출하는 앱 진입점
  - `@app.tool()` : UI가 호출하는 backend tool
  로 나누는 패턴을 사용합니다.

> Colab 한계:
> - 진짜 MCP host(예: ChatGPT/Claude host) 안의 **대화 내 렌더링**은 Colab에서 직접 재현할 수 없습니다.
> - 대신 이 노트북에서는
>   1) **in-memory client로 앱 tool 호출**
>   2) **Prefab HTML 직접 미리보기**
>   3) **FastMCPApp의 entry/backend 구조 체험**
>   방식으로 Apps를 학습합니다.


## 실행 순서

1. **FastMCP Apps 설치**
2. **기본 import**
3. **예제 1: `@mcp.tool(app=True)` 로 간단한 대시보드 앱 만들기**
4. **in-memory client로 앱 tool 호출**
5. **Colab 안에서 HTML 미리보기**
6. **예제 2: `FastMCPApp` 으로 Notes 앱 만들기**
7. **entry-point UI와 backend tool 구조 확인**
8. **Notes 앱 HTML 미리보기**
9. **backend tool 직접 호출로 데이터 갱신 확인**
10. **실습 정리**

권장:
- 메뉴에서 **런타임 → 모두 실행**
- 또는 위에서 아래로 한 셀씩 실행


## 왜 이런 구조인가?

FastMCP Apps 문서에 따르면, 대부분의 앱은 먼저 **Prefab Apps** 부터 시작합니다.  
즉, **tool에 `app=True`를 주고 Prefab component를 반환**하면 사용자는 JSON 대신 인터랙티브 UI를 보게 됩니다.  
그리고 **서버 상호작용이 많은 경우**에는 `FastMCPApp`을 사용해 **모델용 진입점(`@app.ui()`)**과 **UI 전용 backend tool(`@app.tool()`)**을 분리하는 패턴이 권장됩니다.  
이 노트북은 그 두 방식을 모두 보여줍니다.


In [ ]:
# 1) 설치
!pip -q install -U "fastmcp[apps]"


In [ ]:
# 2) 기본 import
from fastmcp import FastMCP, Client, FastMCPApp
from prefab_ui.app import PrefabApp
from prefab_ui.components import Column, Heading, Text, Form, Input, Button, ForEach, Row, Badge, Separator
from prefab_ui.components.charts import BarChart, ChartSeries
from prefab_ui.actions.mcp import CallTool
from prefab_ui.actions import SetState, ShowToast
from prefab_ui.rx import RESULT
from IPython.display import HTML, display
import json

print("import 완료")


## 1) 예제 1: `@mcp.tool(app=True)` 로 간단한 대시보드 앱 만들기

FastMCP Apps 문서에서 가장 먼저 권장하는 패턴입니다.

- `@mcp.tool(app=True)`
- 반환 타입은 `PrefabApp`
- 사용자는 텍스트 대신 UI를 보게 됨

여기서는 연도별 분기 매출 차트를 보여주는 아주 작은 앱을 만듭니다.


In [ ]:
mcp_dashboard = FastMCP(
    "DashboardDemo",
    instructions="Provides a simple Prefab dashboard app."
)

@mcp_dashboard.tool(app=True)
def revenue_chart(year: int = 2025) -> PrefabApp:
    """연도별 분기 매출을 인터랙티브 차트로 보여줍니다."""
    data = [
        {"quarter": "Q1", "revenue": 42000},
        {"quarter": "Q2", "revenue": 51000},
        {"quarter": "Q3", "revenue": 47000},
        {"quarter": "Q4", "revenue": 63000},
    ]

    with Column(gap=4, css_class="p-6") as view:
        Heading(f"{year} Revenue Dashboard")
        Text("FastMCP Apps의 가장 간단한 시작 예제입니다.")
        BarChart(
            data=data,
            series=[ChartSeries(data_key="revenue", label="Revenue")],
            x_axis="quarter",
        )

    return PrefabApp(view=view)

print("예제 1 서버 정의 완료")


## 2) in-memory client로 앱 tool 호출

Colab에서는 `Client(mcp_dashboard)` 로 바로 tool을 호출할 수 있습니다.

관찰 포인트:
- tool 결과 `content`는 보통 `[Rendered Prefab UI]` 형태로 보입니다.
- 실제 UI 구조는 `data` 안에 Prefab JSON 형태로 들어 있습니다.


In [ ]:
def obj_to_dict(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if isinstance(obj, dict):
        return obj
    return str(obj)

async def run_dashboard_tool_demo():
    client = Client(mcp_dashboard)

    async with client:
        print("=== list_tools ===")
        tools = await client.list_tools()
        for t in tools:
            print(json.dumps({
                "name": getattr(t, "name", None),
                "description": getattr(t, "description", None)
            }, ensure_ascii=False, indent=2))

        print("\n=== call_tool(revenue_chart) ===")
        result = await client.call_tool("revenue_chart", {"year": 2025})
        print("content:", getattr(result, "content", None))
        print("data keys:", list((getattr(result, "data", {}) or {}).keys()))
        print("\nstructured data preview:")
        print(json.dumps(getattr(result, "data", None), ensure_ascii=False, indent=2)[:1200])

await run_dashboard_tool_demo()


## 3) Colab 안에서 HTML 미리보기

`PrefabApp.html()` 을 이용하면 Colab에서도 앱 모양을 어느 정도 직접 볼 수 있습니다.

주의:
- 이 미리보기는 **정적/클라이언트 측 렌더링** 확인용입니다.
- MCP host 안에서의 완전한 앱 수명주기와 동일하지는 않습니다.


In [ ]:
dashboard_app = revenue_chart(2025)
display(HTML(dashboard_app.html()))


## 4) 예제 2: `FastMCPApp` 으로 Notes 앱 만들기

문서에 따르면, 서버 상호작용이 많은 앱은 `FastMCPApp` 패턴이 적합합니다.

여기서는:
- `@app.ui()` : Notes 앱을 여는 진입점
- `@app.tool()` : 노트를 저장하거나 조회하는 backend tool

구조로 구현합니다.


In [ ]:
notes_db = [
    {"title": "MCP", "body": "Host, Client, Server 구조를 가진다."},
    {"title": "Apps", "body": "UI를 대화 안에 렌더링할 수 있다."},
]

notes_app = FastMCPApp("NotesApp")

@notes_app.tool()
def add_note(title: str, body: str) -> list[dict]:
    """노트를 저장하고 전체 목록을 반환합니다."""
    notes_db.append({"title": title, "body": body})
    return list(notes_db)

@notes_app.tool(model=True)
def list_notes() -> list[dict]:
    """노트 목록을 반환합니다. 모델과 UI 모두 호출 가능하도록 노출합니다."""
    return list(notes_db)

@notes_app.ui()
def notes_ui() -> PrefabApp:
    """Notes 앱을 엽니다."""
    with Column(gap=6, css_class="p-6") as view:
        Heading("Notes")
        Text("FastMCPApp 예제: UI 진입점과 backend tool을 분리합니다.")

        with ForEach("notes") as note:
            with Row(gap=2):
                Text(note.title)
                Badge(note.body)

        Separator()

        with Form(
            on_submit=CallTool(
                add_note,
                on_success=[
                    SetState("notes", RESULT),
                    ShowToast("Saved!", variant="success"),
                ],
            )
        ):
            Input(name="title", label="Title", required=True)
            Input(name="body", label="Body", required=True)
            Button("Save")

    return PrefabApp(view=view, state={"notes": list(notes_db)})

mcp_notes = FastMCP("NotesServer", providers=[notes_app])

print("예제 2 서버 정의 완료")


## 5) entry-point UI와 backend tool 구조 확인

FastMCP Apps 문서에서는:
- `@app.ui()` 는 모델이 보는 **entry point**
- `@app.tool()` 는 UI가 호출하는 **backend**
로 설명합니다.

이번 셀에서는 실제 tool 목록과 UI 반환 결과를 확인합니다.


In [ ]:
async def run_notes_app_demo():
    client = Client(mcp_notes)

    async with client:
        print("=== list_tools ===")
        tools = await client.list_tools()
        for t in tools:
            print(json.dumps({
                "name": getattr(t, "name", None),
                "description": getattr(t, "description", None)
            }, ensure_ascii=False, indent=2))

        print("\n=== call_tool(notes_ui) ===")
        ui_result = await client.call_tool("notes_ui", {})
        print("content:", getattr(ui_result, "content", None))
        print("data keys:", list((getattr(ui_result, "data", {}) or {}).keys()))

        print("\n=== call_tool(list_notes) ===")
        list_result = await client.call_tool("list_notes", {})
        print("data:", getattr(list_result, "data", None))

await run_notes_app_demo()


## 6) Notes 앱 HTML 미리보기

`notes_ui().html()` 로 Colab 안에서 앱 화면을 볼 수 있습니다.

주의:
- 폼에서 실제 `CallTool(...)` 이 동작하는 full MCP host는 Colab이 아닙니다.
- 하지만 UI 구조와 backend 분리 개념은 충분히 확인할 수 있습니다.


In [ ]:
display(HTML(notes_ui().html()))


## 7) backend tool 직접 호출로 데이터 갱신 확인

UI 안에서는 `CallTool(add_note, ...)` 가 호출될 자리입니다.  
Colab에서는 backend tool을 client에서 직접 호출해 같은 효과를 확인합니다.


In [ ]:
async def simulate_backend_updates():
    client = Client(mcp_notes)

    async with client:
        print("=== add_note 호출 ===")
        add_result = await client.call_tool(
            "add_note",
            {"title": "Colab Demo", "body": "직접 backend tool을 호출해 데이터 갱신을 확인합니다."}
        )
        print("data:", getattr(add_result, "data", None))

        print("\n=== list_notes 재조회 ===")
        list_result = await client.call_tool("list_notes", {})
        print(json.dumps(getattr(list_result, "data", None), ensure_ascii=False, indent=2))

await simulate_backend_updates()


In [ ]:
# 갱신된 Notes 앱 HTML 다시 보기
display(HTML(notes_ui().html()))


## 실습 정리

이번 실습에서 확인한 핵심:

1. **가장 쉬운 시작**은 `@mcp.tool(app=True)` + `PrefabApp` 반환이다.
2. 앱 tool을 client로 호출하면 `data` 안에 Prefab UI 구조가 들어간다.
3. `PrefabApp.html()` 로 Colab에서도 UI를 미리 볼 수 있다.
4. 상호작용이 많은 앱은 `FastMCPApp` 구조가 적합하다.
5. `@app.ui()` 는 앱 진입점, `@app.tool()` 는 UI backend 역할을 한다.
6. Colab은 full MCP host는 아니므로, UI 내부 `CallTool`의 완전한 런타임은 제한적이다.
7. 그래도 **entry point / backend / state / UI 구조**는 충분히 학습할 수 있다.

다음 확장 아이디어:
- `fastmcp dev apps server.py` 로 로컬 브라우저 미리보기
- 실제 MCP host에서 앱 렌더링 확인
- `CallTool` 결과를 state에 연결하는 다단계 앱 실습
